# LightGBM Classifier - Current Review Schema

Strict current-schema notebook. It requires taxonomy review columns such as `workflow_status`, `morphology_primary`, and `physical_primary`; it does not derive labels from legacy `event_class` or old physical-family columns.

In [ ]:
from pathlib import Path

import pandas as pd

from malca.meta_analysis.ml.review_lightgbm import (
    CURRENT_TAXONOMY_TARGET_COLUMNS,
    TrainingConfig,
    label_audit,
    load_current_schema_training_table,
    train_review_models,
)

REVIEW_DB = Path("output/review/review.db")
FLAT_LIGHTCURVE_DIR = Path("output/review/bundle_assets/lightcurves")
OUTPUT_DIR = Path("output/ml/current_schema")
LIGHTCURVE_FEATURE_CACHE = OUTPUT_DIR / "lightcurve_features.parquet"

TARGET_COLUMNS = list(CURRENT_TAXONOMY_TARGET_COLUMNS)
INCLUDE_LIGHTCURVE_FEATURES = True
MAX_LIGHTCURVES = None

CONFIG = TrainingConfig(
    random_state=42,
    test_size=0.2,
    cv_folds=5,
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    n_jobs=1,
)


## Load Review Rows And Light-Curve Features

In [ ]:
flat_lc_dir = FLAT_LIGHTCURVE_DIR if FLAT_LIGHTCURVE_DIR.exists() else None
table = load_current_schema_training_table(
    REVIEW_DB,
    flat_lightcurve_dir=flat_lc_dir,
    include_lightcurve_features=INCLUDE_LIGHTCURVE_FEATURES,
    lightcurve_feature_cache=LIGHTCURVE_FEATURE_CACHE,
    only_reviewed=True,
    max_lightcurves=MAX_LIGHTCURVES,
)

print(f"Loaded {len(table):,} reviewed rows and {len(table.columns):,} columns")
print(f"Review DB: {REVIEW_DB}")
print(f"Flat light-curve dir: {flat_lc_dir}")
display(table.head())


## Label Audit

In [ ]:
audits = label_audit(table, TARGET_COLUMNS)
for target, counts in audits.items():
    print(f"\n{target}")
    display(counts.rename("n").to_frame())


## Train, Holdout-Test, And Cross-Validate

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results = train_review_models(
    table,
    TARGET_COLUMNS,
    output_dir=OUTPUT_DIR,
    config=CONFIG,
)

for target, result in results.items():
    print(f"\n{target}: {result.n_rows:,} rows, {result.n_features:,} features")
    print("classes:", result.class_counts)
    print("holdout:", {k: v for k, v in result.holdout_metrics.items() if k != "classification_report"})
    if not result.cv_metrics.empty:
        display(result.cv_metrics)


## Feature Importance

In [ ]:
for target, result in results.items():
    booster = result.model.booster_
    importance = pd.DataFrame(
        {
            "feature": result.feature_columns,
            "gain": booster.feature_importance(importance_type="gain"),
            "split": booster.feature_importance(importance_type="split"),
        }
    ).sort_values("gain", ascending=False)
    print(f"\n{target}")
    display(importance.head(30))
